# Silver – Olist
Opschonen van bronze-data: juiste datatypes, dubbele rijen eruit, ongeldige rijen eruit.


In [0]:
from pyspark.sql import functions as F

CATALOG = "bazar9_dbw"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"

def read_bronze(name):
    """Lees een bronze-tabel."""
    return spark.table(f"{BRONZE}.{name}")

def tc(col, datatype):
    """Zet tekst om naar een datatype. Lukt dat niet, dan wordt het leeg (NULL)."""
    return F.expr(f"try_cast({col} AS {datatype})")

def ts(col):
    """Zet tekst om naar datum + tijd."""
    return F.expr(f"try_to_timestamp({col})")

def save_silver(df, name):
    """Sla op als silver-tabel en toon het aantal rijen."""
    (
        df.withColumn("_processed_at", F.current_timestamp())
        .write.mode("overwrite")
        .option("overwriteSchema", True)
        .saveAsTable(f"{SILVER}.{name}")
    )
    print(f"{name}: {spark.table(f'{SILVER}.{name}').count():,} rijen")

In [0]:
orders = (
    read_bronze("orders")
    .select(
        "order_id",
        "customer_id",
        "order_status",
        ts("order_purchase_timestamp").alias("purchased_at"),
        ts("order_approved_at").alias("approved_at"),
        ts("order_delivered_carrier_date").alias("delivered_carrier_at"),
        ts("order_delivered_customer_date").alias("delivered_customer_at"),
        ts("order_estimated_delivery_date").alias("estimated_delivery_at"),
        "_ingested_at",
    )
    .filter(F.col("order_id").isNotNull())
    .dropDuplicates(["order_id"])
)
save_silver(orders, "orders")

In [0]:
for schema in ["bronze", "silver"]:
    print(f"--- {schema} ---")
    tables = spark.sql(f"SHOW TABLES IN bazar9_dbw.{schema}").collect()
    if not tables:
        print("(leeg)")
    for t in tables:
        n = spark.table(f"bazar9_dbw.{schema}.{t.tableName}").count()
        print(f"{t.tableName}: {n:,} rijen")